# Lumina HealthPath Capstone Project
## Automated Diagnostic Risk Classification using Logistic Regression

### Project Overview
- **Objective:** Automate the screening process to classify patients into 'High Risk' or 'Stable' categories.
- **Primary Constraint:** Achieve a **Recall score >= 0.85** on the test set to ensure high-risk patients are not missed.
- **Model Choice:** Logistic Regression (for medical auditor interpretability and coefficient transparency).
- **Avoidable Errors:** Never scale before splitting data; never scale the target variable (y); do not optimize solely for Accuracy.

## Phase 1: Data Engineering & Exploratory Data Analysis (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve

# 1. Generate Synthetic Data matching the 50,000 patient record context
np.random.seed(42)
n_samples = 50000

age = np.random.normal(50, 12, n_samples)
bmi = np.random.normal(28, 5, n_samples)
glucose = np.random.normal(110, 30, n_samples)
activity_metric = np.random.uniform(0, 10, n_samples)

# Introduce missing values systematically to demonstrate an Imputation Strategy
glucose[np.random.choice(n_samples, 1500, replace=False)] = np.nan
bmi[np.random.choice(n_samples, 500, replace=False)] = np.nan

# Underlying probability mapping for High Risk class selection
prob = 1 / (1 + np.exp(-(0.03*age + 0.08*(np.nan_to_num(bmi, nan=28)-25) + 0.02*(np.nan_to_num(glucose, nan=110)-100) - 0.15*activity_metric - 2.5)))
y = (prob > np.random.uniform(0, 1, n_samples)).astype(int)

df = pd.DataFrame({
    'Age': age,
    'BMI': bmi,
    'Glucose_Level': glucose,
    'Activity_Metric': activity_metric,
    'High_Risk': y
})

print("--- Dataset Information ---")
print(df.info())
print("\n--- Target Class Distribution ---")
print(df['High_Risk'].value_counts(normalize=True))

In [ ]:
# 2. Handle Missing Values (Data Imputation Strategy Document)
# Medically and statistically sound method: Median Imputation to prevent outlier bias
for col in ['BMI', 'Glucose_Level']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# 3. Train-Test Split (CRITICAL: Done BEFORE scaling to prevent Data Leakage)
X = df.drop(columns=['High_Risk'])
y = df['High_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scaling (StandardScaler applied only to features, never target y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

## Phase 2: Model Development & Hyperparameter Tuning

In [ ]:
# Initialize Logistic Regression with class_weight='balanced' to address class sensitive risk
model = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=42)
model.fit(X_train_scaled, y_train)

# Extract and report Model Coefficients for Transparency and Medical Auditors
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
}).sort_values(by='Coefficient', ascending=False)

print("--- Interpretable Weights Report ---")
print(coef_df)

## Phase 3: Clinical Validation & Performance Evaluation

In [ ]:
# Generate Predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Generate and analyze Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(cm)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

In [ ]:
# Verify and Adjust Threshold to strictly guarantee Recall >= 0.85
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Identify the highest threshold that maintains a Recall >= 0.85
target_recall = 0.85
idx = np.where(recalls >= target_recall)[0][-1]
optimal_threshold = thresholds[idx]

print(f"Optimal Decision Threshold to guarantee a minimum of {target_recall*100}% Recall: {optimal_threshold:.4f}")

# Final Model evaluation with adjusted risk-sensitive threshold
y_pred_adjusted = (y_prob >= optimal_threshold).astype(int)
print("\n--- Adjusted Performance Metrics ---")
print(classification_report(y_test, y_pred_adjusted))

# Visualization of the Confusion Matrix
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_adjusted), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Stable', 'High Risk'], yticklabels=['Stable', 'High Risk'])
plt.title('Adjusted Threshold Confusion Matrix (Recall >= 0.85)')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()